# Project 01 (basic) — Point operations & histogram equalization

**Module 12 — Image Processing** · Format: **Jupyter notebook** (`point_ops_histogram.ipynb`)

The simplest image operations are **point operations**: every pixel is recomputed
*independently* of its neighbours. Despite their simplicity they are behind brightness,
contrast, gamma and **histogram equalization** — one of the nicest small algorithms of image
processing. Here you build them by hand and make their effect visible on the histogram.

Core questions (script section 1):
- How do linear point operations and **gamma** change an image?
- What does the **histogram** describe, and how does the **equalization via the CDF** stretch
  the contrast?

> **Plenty of instruction, no training, only `numpy`.** Runs in seconds.


## Setup

Requires `numpy`, `matplotlib`, `Pillow` (repo `requirements.txt`). The example image
(*Grace Hopper*) is included in matplotlib — no download.

```bash
source ../../../../.venv/bin/activate
jupyter lab      # or open the notebook in VS Code, kernel = repo .venv
```


## Part A — Image & histogram *(given)*

We load a grayscale image (values $0$–$255$) and show it with its **histogram** — the
distribution of the gray values.


In [ ]:
# ---- Image + histogram (given) -------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cbook as cbook
from PIL import Image

with cbook.get_sample_data("grace_hopper.jpg") as f:
    img = np.asarray(Image.open(f).convert("L"), dtype=np.uint8)   # grayscale 0..255
print("Image:", img.shape, "dtype", img.dtype, "min/max", img.min(), img.max())

def show_with_hist(images, titles):
    n = len(images)
    fig, ax = plt.subplots(2, n, figsize=(4 * n, 6))
    ax = np.array(ax).reshape(2, n)          # a 2D axis matrix even for n=1
    for k, (im, t) in enumerate(zip(images, titles)):
        ax[0, k].imshow(im, cmap="gray", vmin=0, vmax=255); ax[0, k].set_title(t); ax[0, k].axis("off")
        ax[1, k].hist(im.ravel(), bins=256, range=(0, 255), color="steelblue")
        ax[1, k].set_xlim(0, 255); ax[1, k].set_yticks([])
    plt.tight_layout(); plt.show()

show_with_hist([img], ["Original"])

### Task 1 — Linear point operation & gamma

Implement two point operations $s=T(r)$ (script 1.2). Mind the **clipping** to $[0,255]$ and
`uint8`.

- **Linear** (contrast $a$, brightness $b$): $s = a\,r + b$.
- **Gamma:** $s = 255\,(r/255)^{\gamma}$ — $\gamma<1$ brightens dark areas.

**Your task (`# TODO`):** fill in `linear(img, a, b)` and `gamma(img, g)`.


In [ ]:
# ---- TASK 1: point operations --------------------------------------------
def linear(im, a, b):
    # TODO: s = a*r + b, clip to [0,255], return as uint8
    raise NotImplementedError("Task 1a: implement linear")

def gamma(im, g):
    # TODO: s = 255*(r/255)**g, clip, uint8
    raise NotImplementedError("Task 1b: implement gamma")

dark = linear(img, 1.0, -60)
contr = linear(img, 1.6, -60)
g_bright = gamma(img, 0.5)
show_with_hist([img, dark, contr, g_bright],
               ["Original", "Brighter/darker (b=-60)", "Contrast (a=1.6)", "Gamma 0.5"])

### Task 2 — Histogram equalization by hand

**Histogram equalization** uses the **cumulative distribution** (CDF) as the mapping
(script 1.3):
$$s = T(r) = \operatorname{round}\!\big((L-1)\cdot \text{cdf}(r)\big),\quad L=256.$$

**Your task (`# TODO`):** implement `equalize(img)`:
1. the histogram of the 256 gray values (`np.bincount(im.ravel(), minlength=256)`);
2. normalize to $p(k)$, accumulate to the `cdf`;
3. the mapping `T = round(255 * cdf)` (as `uint8`), then apply `T[img]`.


In [ ]:
# ---- TASK 2: histogram equalization --------------------------------------
def equalize(im):
    # TODO: histogram -> cdf -> T=round(255*cdf) (uint8) -> return T[im]
    raise NotImplementedError("Task 2: implement equalize")

low_contrast = linear(img, 0.4, 90)
equalized = equalize(low_contrast)
show_with_hist([low_contrast, equalized, equalize(img)],
               ["Low contrast", "Equalized", "Equalize(original)"])

### Task 3 — Make the CDF visible & self-check

**Your task (`# TODO`):** plot the **CDF** of the low-contrast image before and after
equalization. After equalization the CDF should be **approximately a straight line** (the
diagonal) — that is the graphical signature of the uniform distribution.


In [ ]:
# ---- TASK 3: CDF before/after equalization -------------------------------
def cdf_of(im):
    h = np.bincount(im.ravel(), minlength=256).astype(np.float64)
    return np.cumsum(h) / h.sum()

# TODO: plot cdf_of(low_contrast) and cdf_of(equalized), plus the ideal diagonal
raise NotImplementedError("Task 3: plot the CDF before/after equalization")

# self-check: equalization uses (almost) the full value range
print("Value range low contrast:", low_contrast.min(), "-", low_contrast.max())
print("Value range equalized:", equalized.min(), "-", equalized.max())

## Reflection (short, written)

1. **Gamma:** why does $\gamma<1$ brighten *dark* areas more than bright ones? (Think of the
   course of $x^\gamma$ on $[0,1]$.)
2. **Equalization:** why does the CDF mapping make the histogram (approximately) uniform, and
   why is the CDF afterwards almost a straight line?
3. **Limits:** histogram equalization is *global*. On an image with very bright and very dark
   regions it can overdrive locally. Which idea (keyword *adaptive*, e.g. CLAHE) would limit
   that locally?
4. **Point operations:** why can *no* point operation remove noise or sharpen edges? (What
   would one need instead — an outlook on project 02?)

> Write down your answers first, then compare them with the **reference answers at the end of
> the solution** in the folder `solution/`.

> **Reference:** after equalization the image uses (almost) the full range 0–255, the
> histogram is wider/flatter, the CDF is close to the diagonal. Gamma 0.5 visibly lifts shadow
> detail.
